# 2. Conventional Approaches to Phase Identification

In [ ]:
# @title Environment Setup
!pip install pymatgen numpy matplotlib -q
# Installation should take > 1 minute to complete
# You may see a dependency conflict warning about 'requests'
# This is harmless and can be ignored
print("Packages installed")

In [ ]:
# @title Load Tutorial Data
import os

REPO = "MRS_CH08_Tutorial"
REPO_URL = "https://github.com/Szymanski-Group/MRS_CH08_Tutorial.git"

if os.path.basename(os.getcwd()) == REPO:
    print("Data already present")
elif os.path.exists(REPO):
    os.chdir(REPO)
    print("Data already present")
else:
    !git clone {REPO_URL} -q
    os.chdir(REPO)
    print("Data loaded successfully")


## 2a) Search Match

Here we detect peaks in experimental patterns and rank candidate phases using FoM-style line matching scores.

In [ ]:
import csv
import glob
from pathlib import Path

from IPython.display import Image, display

# We'll import some pre-written scripts from this repo
# You can find them in: MRS_CH08_Tutorial/tutorial_utils
from tutorial_utils.conventional import search_match as sm
from tutorial_utils.sections import conventional_search_match as vis_sm


def create_search_match_demo(
    top_k_to_print=3,
    max_experiment_patterns=4,
    match_tolerance_deg=0.25,
    min_peak_distance_deg=0.22,
    num_obs_lines_for_fom=20,
):
    """2a) Peak search-match with explicit step-by-step flow."""

    # Keep only the user-facing knobs above; everything else uses hidden defaults.
    EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
    REFERENCE_DIR = Path("data/reference_structures")
    OUTPUT_DIR = Path("outputs/conventional/search_match")
    MIN_ANGLE, MAX_ANGLE = 10.0, 80.0

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    # Match plotting range to analysis range so there is one clear angular window.
    vis_sm.OUTPUT_DIR = OUTPUT_DIR
    vis_sm.TOP_K_TO_PRINT = top_k_to_print
    vis_sm.PLOT_MIN_ANGLE = MIN_ANGLE
    vis_sm.PLOT_MAX_ANGLE = MAX_ANGLE

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    if max_experiment_patterns is not None:
        exp_files = exp_files[:max_experiment_patterns]

    # Step 0: build reference stick library once.
    refs = sm.load_reference_library(
        sorted(REFERENCE_DIR.glob("*.cif")),
        min_angle=MIN_ANGLE,
        max_angle=MAX_ANGLE,
    )

    all_rows = []
    print("\n=== Peak Search-Match Demo (de Wolff + Smith-Snyder) ===")
    print(f"Experimental patterns: {len(exp_files)}")
    print(f"Reference phases:      {len(refs)}")

    for exp_file in exp_files:
        pattern_name = exp_file.stem

        # Step 1: load + normalize experimental profile.
        tt, intensity = sm.load_pattern(exp_file, min_angle=MIN_ANGLE, max_angle=MAX_ANGLE)

        # Step 2: detect candidate peak positions.
        _, obs_peaks = sm.detect_peaks(
            tt,
            intensity,
            min_peak_distance_deg=min_peak_distance_deg,
        )

        # Step 3: rank phases with FoM calculations.
        by_dewolff, by_smith = sm.rank_phases(
            obs_peaks,
            refs,
            num_obs_lines_for_fom=num_obs_lines_for_fom,
            match_tolerance_deg=match_tolerance_deg,
        )

        print(f"\n--- {pattern_name} ---")
        print(f"Detected peaks: {len(obs_peaks)}")
        vis_sm.print_rank_table(pattern_name, by_dewolff, "de_wolff", "de Wolff")
        vis_sm.print_rank_table(pattern_name, by_smith, "smith_snyder", "Smith-Snyder")

        # Step 4: visualize best matches.
        vis_sm.plot_summary(pattern_name, tt, intensity, obs_peaks, by_dewolff[0], by_smith[0], refs)

        for row in by_dewolff:
            all_rows.append({"pattern": pattern_name, "phase": row["phase"], **{k: row[k] for k in row if k != "phase"}})

    csv_file = OUTPUT_DIR / "all_pattern_rankings.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=[
                "pattern",
                "phase",
                "de_wolff",
                "smith_snyder",
                "n_used",
                "n_match",
                "n_possible",
                "mean_delta_2theta",
            ],
        )
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nSaved ranking table: {csv_file}")
    return all_rows


## Run the Demo


In [ ]:
# @title Run Search-Match Demo
create_search_match_demo()

# Try on your own:
# 1) Loosen/tighten line matching tolerance and compare rankings.
# create_search_match_demo(match_tolerance_deg=0.18)
# create_search_match_demo(match_tolerance_deg=0.32)
#
# 2) Use fewer or more observed peaks in the FoM calculation.
# create_search_match_demo(num_obs_lines_for_fom=10)
# create_search_match_demo(num_obs_lines_for_fom=30)
#
# 3) Change peak-distance filtering to see how peak picking affects FoM.
# create_search_match_demo(min_peak_distance_deg=0.12)
# create_search_match_demo(min_peak_distance_deg=0.35)


## Example Output


In [ ]:
for p in sorted(glob.glob("outputs/conventional/search_match/*_summary.png"))[:3]:
    display(Image(p))

## Next Steps
Continue to the next section below in this notebook.

## 2b) Profile Correlation


In [ ]:
import csv
import glob
from pathlib import Path

from IPython.display import Image, display

from tutorial_utils.conventional import profile_correlation as pc
from tutorial_utils.sections import conventional_profile_correlation as vis_pc


def create_profile_correlation_demo(
    top_k_to_print=3,
    max_experiment_patterns=4,
    fwhm=0.30,
    gauss_frac=0.2,
):
    """2b) Full-profile correlation with explicit step-by-step flow."""

    # Keep only key user-facing knobs above; use hidden defaults for preprocessing details.
    EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
    REFERENCE_DIR = Path("data/reference_structures")
    OUTPUT_DIR = Path("outputs/conventional/profile_correlation")
    MIN_ANGLE, MAX_ANGLE = 10.0, 80.0

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    vis_pc.OUTPUT_DIR = OUTPUT_DIR
    vis_pc.TOP_K_TO_PRINT = top_k_to_print
    vis_pc.FWHM = fwhm
    vis_pc.GAUSS_FRAC = gauss_frac

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    if max_experiment_patterns is not None:
        exp_files = exp_files[:max_experiment_patterns]

    # Step 0: load reference sticks once.
    ref_lib = pc.load_reference_stick_library(
        sorted(REFERENCE_DIR.glob("*.cif")),
        min_angle=MIN_ANGLE,
        max_angle=MAX_ANGLE,
    )

    all_rows = []
    print("\n=== Full-Profile Correlation Demo (Pearson + Cosine) ===")
    print(f"Experimental patterns: {len(exp_files)}")
    print(f"Reference phases:      {len(ref_lib)}")

    for exp_file in exp_files:
        pattern_name = exp_file.stem

        # Step 1: preprocess experimental profile.
        two_theta, exp_profile = pc.load_experimental_profile(
            exp_file,
            min_angle=MIN_ANGLE,
            max_angle=MAX_ANGLE,
        )

        # Step 2: simulate each candidate and compute similarity metrics.
        by_pearson, by_cosine, simulated_profiles = pc.rank_phases(
            exp_profile,
            two_theta,
            ref_lib,
            fwhm=fwhm,
            gauss_frac=gauss_frac,
        )

        print(f"\n--- {pattern_name} ---")
        vis_pc.print_rank_table(pattern_name, by_pearson, "pearson", "Pearson")
        vis_pc.print_rank_table(pattern_name, by_cosine, "cosine", "Cosine")

        # Step 3: visualize best matches.
        vis_pc.plot_summary(pattern_name, two_theta, exp_profile, by_pearson[0], by_cosine[0], simulated_profiles)

        for row in by_pearson:
            all_rows.append({"pattern": pattern_name, "phase": row["phase"], "pearson": row["pearson"], "cosine": row["cosine"]})

    csv_file = OUTPUT_DIR / "all_pattern_profile-correlations.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["pattern", "phase", "pearson", "cosine"])
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nSaved ranking table: {csv_file}")
    return all_rows


This method compares full simulated and observed profiles, instead of matching only discrete peak positions.


## Run the Demo
Run the step-by-step function in the next cell.


In [ ]:
# @title Run Profile-Correlation Demo
create_profile_correlation_demo()

# Try on your own:
# 1) Increase/decrease broadening and see ranking stability.
# create_profile_correlation_demo(fwhm=0.20)
# create_profile_correlation_demo(fwhm=0.55)
#
# 2) Sweep Gaussian-vs-Lorentzian mixing.
# create_profile_correlation_demo(gauss_frac=0.0)
# create_profile_correlation_demo(gauss_frac=0.8)
#
# 3) Focus on one pattern at a time while debugging.
# create_profile_correlation_demo(max_experiment_patterns=1)


## Example Output


In [ ]:
for p in sorted(glob.glob("outputs/conventional/profile_correlation/*_profile-correlation.png"))[:3]:
    display(Image(p))

## Next Steps
Continue to the next section below in this notebook.

## 2c) Sequential Rietveld-Style Refinement


In [ ]:
import csv
import glob
from pathlib import Path

from IPython.display import Image, display

from tutorial_utils.conventional import rietveld as rv
from tutorial_utils.sections import conventional_rietveld as vis_rv


def run_background_refinement_step(two_theta, y_obs, structure, calculator, min_angle, max_angle, background_degree, fwhm_init):
    """Step 1: refine background with fixed lattice + fixed width."""
    return rv.refine_background_step(
        two_theta,
        y_obs,
        structure,
        calculator,
        min_angle=min_angle,
        max_angle=max_angle,
        background_degree=background_degree,
        fwhm_init=fwhm_init,
    )


def run_peak_position_refinement_step(two_theta, y_obs, structure, calculator, y_bg, min_angle, max_angle, fwhm_init):
    """Step 2: refine peak positions via lattice-scale refinement."""
    return rv.refine_lattice_step(
        two_theta,
        y_obs,
        structure,
        calculator,
        y_bg,
        min_angle=min_angle,
        max_angle=max_angle,
        fwhm_init=fwhm_init,
    )


def run_peak_width_refinement_step(two_theta, y_obs, peak_pos, peak_int, y_bg, fwhm_init):
    """Step 3: refine global peak width with lattice/background fixed."""
    return rv.refine_width_step(
        two_theta,
        y_obs,
        peak_pos,
        peak_int,
        y_bg,
        fwhm_init=fwhm_init,
    )


def assemble_rietveld_row(phase, y_obs, step1, step2, step3):
    """Collect key outputs from all three sequential refinement steps."""
    return {
        "phase": phase,
        "rwp": rv.compute_rwp(y_obs, step3["y_fit"]),
        "pearson": rv.pearson_corr(y_obs, step3["y_fit"]),
        "scales": step2["scales"],
        "fwhm": step3["fwhm"],
        "y_bg": step1["y_bg"],
        "y_fit_step1": step1["y_fit"],
        "y_fit_step2": step2["y_fit"],
        "y_fit_final": step3["y_fit"],
    }


def create_rietveld_demo(
    top_k_to_print=3,
    patterns_to_run=("TiO2", "ZrO2"),
    background_degree=6,
    fwhm_init=0.30,
):
    """2c) Sequential Rietveld-style refinement with explicit step-by-step flow."""

    # Keep only key user-facing knobs above; use hidden defaults for optimizer settings.
    EXPERIMENT_DIR = Path("data/exp_patterns/one_phase")
    REFERENCE_DIR = Path("data/reference_structures")
    OUTPUT_DIR = Path("outputs/conventional/rietveld")
    MIN_ANGLE, MAX_ANGLE = 10.0, 80.0

    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

    vis_rv.OUTPUT_DIR = OUTPUT_DIR
    vis_rv.TOP_K_TO_PRINT = top_k_to_print

    exp_files = sorted(EXPERIMENT_DIR.glob("*.xy"))
    if patterns_to_run is not None:
        keep = set(patterns_to_run)
        exp_files = [f for f in exp_files if f.stem in keep]

    # Step 0: load candidate structures and create calculator.
    structures = rv.load_reference_structures(sorted(REFERENCE_DIR.glob("*.cif")))
    calculator = rv.make_calculator()

    all_rows = []
    print("\n=== Sequential Rietveld-Style Demo ===")
    print("Refinement order: background -> peak position (lattice) -> peak width")
    print(f"Experimental patterns: {len(exp_files)}")
    print(f"Reference phases:      {len(structures)}")

    for exp_file in exp_files:
        pattern_name = exp_file.stem

        # Step 1: preprocess experimental profile.
        two_theta, y_obs = rv.load_experimental_profile(
            exp_file,
            min_angle=MIN_ANGLE,
            max_angle=MAX_ANGLE,
        )

        # Step 2: run sequential refinement per candidate phase.
        rows = []
        for phase, structure in structures.items():
            step1 = run_background_refinement_step(
                two_theta,
                y_obs,
                structure,
                calculator,
                min_angle=MIN_ANGLE,
                max_angle=MAX_ANGLE,
                background_degree=background_degree,
                fwhm_init=fwhm_init,
            )
            step2 = run_peak_position_refinement_step(
                two_theta,
                y_obs,
                structure,
                calculator,
                step1["y_bg"],
                min_angle=MIN_ANGLE,
                max_angle=MAX_ANGLE,
                fwhm_init=fwhm_init,
            )
            step3 = run_peak_width_refinement_step(
                two_theta,
                y_obs,
                step2["peak_pos"],
                step2["peak_int"],
                step1["y_bg"],
                fwhm_init=fwhm_init,
            )
            rows.append(assemble_rietveld_row(phase, y_obs, step1, step2, step3))

        # Step 3: rank by final fit quality.
        rows.sort(key=lambda r: r["rwp"])

        print(f"\n--- {pattern_name} ---")
        vis_rv.print_rank_table(pattern_name, rows)

        # Step 4: show each sequential step in the summary plot for the best phase.
        vis_rv.plot_refinement_summary(pattern_name, two_theta, y_obs, rows[0])

        for row in rows:
            s = row["scales"]
            all_rows.append(
                {
                    "pattern": pattern_name,
                    "phase": row["phase"],
                    "rwp": row["rwp"],
                    "pearson": row["pearson"],
                    "a_scale": s[0],
                    "b_scale": s[1],
                    "c_scale": s[2],
                    "fwhm": row["fwhm"],
                }
            )

    csv_file = OUTPUT_DIR / "all_pattern_rietveld-sequential.csv"
    with open(csv_file, "w", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["pattern", "phase", "rwp", "pearson", "a_scale", "b_scale", "c_scale", "fwhm"],
        )
        writer.writeheader()
        writer.writerows(all_rows)

    print(f"\nSaved ranking table: {csv_file}")
    return all_rows


This simplified workflow refines background, lattice scales, and peak width in sequence for each candidate phase.


## Run the Demo
Run the step-by-step function in the next cell.


In [ ]:
# @title Run Sequential Rietveld Demo
create_rietveld_demo()

# Try on your own:
# 1) Change background flexibility.
# create_rietveld_demo(background_degree=4)
# create_rietveld_demo(background_degree=8)
#
# 2) Change initial peak width and see convergence behavior.
# create_rietveld_demo(fwhm_init=0.18)
# create_rietveld_demo(fwhm_init=0.45)
#
# 3) Run one phase at a time and inspect refinement plots.
# create_rietveld_demo(patterns_to_run=("TiO2",))


## Example Output


In [ ]:
for p in sorted(glob.glob("outputs/conventional/rietveld/*_rietveld-sequential.png")):
    display(Image(p))

## Next Steps
Continue to **03 — ML + Deep Learning**: [Open in Colab](https://colab.research.google.com/github/Szymanski-Group/MRS_CH08_Tutorial/blob/main/notebooks/03_ML-Deep-Learning.ipynb)